## Open Tools, Open Sky
### Doing LSST Science with Public Rubin Data & LINCC Frameworks
#### Latest update: Jorge I. Zuluaga

## Template notebook

In [4]:
from util import *
import requests
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
from astroquery.jplsbdb import SBDB
import matplotlib.dates as mdates
from astropy.timeseries import LombScargle
import re
from scipy.signal import find_peaks

# Macros
DEG = np.pi / 180
RAD = 1 / DEG

# Directories
fig_dir='figures/'
data_dir='data/'
!mkdir -p $data_dir $fig_dir

# Reload util.py routines
%load_ext autoreload 
%autoreload 2

# Check that data files exists
get_data(data_dir = data_dir)

Data already exists: data/obs_sbn_X05_full.parquet
Data already exists: data/mpcorb_extended.dat.gz
Data already exists: data/asteroids-with-rotation.csv
Data already exists: data/X05_simplified.csv
Data already exists: data/obs_final_elements_geometry_H.parquet
Data already exists: data/sbdb_physical_properties.csv
Data already exists: data/mpc_designations.csv
Download complete or files already present.


## Build a catalogue

Each `.txt` file is a list of starred phase-curve figures, one filename per line:

`phase_allbands_<year>_<designation>.webp`

That name is the provisional designation with the space replaced by an underscore, so `phase_allbands_2006_ST360.webp` is the provid `2006 ST360`. The same asteroid can appear in more than one list. The catalogue keeps each provid once.

In [5]:
from pathlib import Path

FIGURE_PREFIX = "phase_allbands_"

def provid_from_figure_name(name):
    """Recover the provid from a starred figure filename.

    phase_allbands_2006_ST360.webp -> 2006 ST360
    """
    stem = Path(name.strip()).stem
    if not stem.startswith(FIGURE_PREFIX):
        raise ValueError(f"Unexpected figure name: {name!r}")
    year, designation = stem[len(FIGURE_PREFIX):].split("_", 1)
    return f"{year} {designation}"

def catalogue_from_starred_lists(files):
    """Build a unique provid catalogue from a list of starred .txt files."""
    seen = set()
    provids = []
    n_lines = 0
    for path in files:
        for line in Path(path).read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            n_lines += 1
            provid = provid_from_figure_name(line)
            if provid in seen:
                continue
            seen.add(provid)
            provids.append(provid)
    catalogue = (
        pd.DataFrame({"provid": provids})
        .sort_values("provid")
        .reset_index(drop=True)
    )
    return catalogue, n_lines

starred_files = sorted(Path(data_dir).glob("medellin_workshop_solarsystem-asteroid-starred-*.txt"))
catalogue, n_lines = catalogue_from_starred_lists(starred_files)

out = Path(data_dir) / "medellin-vro-catalogue.csv"
catalogue.to_csv(out, index=False)

print(f"{len(starred_files)} lists, {n_lines} names, {len(catalogue)} unique provids")
print(f"Wrote {out}")
catalogue.head()

7 lists, 611 names, 508 unique provids
Wrote data/medellin-vro-catalogue.csv


,provid
0,1978 UU1
1,1981 EY23
2,1986 PX5
3,1992 EP8
4,1997 JU3


## Overlap with the rotation catalogue

`asteroids-with-rotation.csv` identifies each object by its current `provid` and, when known, an earlier `alt_designation`. A catalogue asteroid is the same object if its provid matches either column.

In [ ]:
vro = pd.read_csv(data_dir + "medellin-vro-catalogue.csv")
rot = pd.read_csv(data_dir + "asteroids-with-rotation.csv")

same_provid = set(vro["provid"]) & set(rot["provid"])
alt_provid = set(rot["alt_designation"].dropna())
same_alt = (set(vro["provid"]) & alt_provid) - same_provid

n_vro = len(vro)
n_shared = len(same_provid) + len(same_alt)
print(f"{n_shared} of {n_vro} catalogue asteroids are in asteroids-with-rotation.csv")
print(f"  {len(same_provid)} match the current provid")
print(f"  {len(same_alt)} match an earlier designation: {', '.join(sorted(same_alt))}")

## Observations in df_set

`df_set` is the asteroid subset of `obs_sbn_X05_full.parquet`: a row whose `provid` matches `YYYY LL[NN]`. Every object in `asteroids-with-rotation.csv` has that form, so its observation count is the number of parquet rows with that `provid`. When the object also has an earlier designation, those rows are counted separately.

In [ ]:
obs_file = data_dir + "obs_sbn_X05_full.parquet"
rot = pd.read_csv(data_dir + "asteroids-with-rotation.csv")

ids = pd.concat([rot["provid"], rot["alt_designation"]]).dropna().astype(str).str.strip()
ids = sorted(set(ids) - {""})
counts = (
    pq.read_table(obs_file, columns=["provid"], filters=[("provid", "in", ids)])
    .to_pandas()["provid"]
    .value_counts()
)

obs_table = rot[["provid", "alt_designation"]].copy()
obs_table["n_obs"] = obs_table["provid"].map(counts).fillna(0).astype(int)
obs_table["n_obs_alt"] = obs_table["alt_designation"].map(counts).fillna(0).astype(int)
obs_table = obs_table.sort_values(["n_obs", "provid"]).reset_index(drop=True)

n_below = int((obs_table["n_obs"] < 100).sum())
print(f"{n_below} of {len(obs_table)} have fewer than 100 observations under the current provid")
print(
    f"n_obs ranges from {obs_table['n_obs'].min()} to {obs_table['n_obs'].max()} "
    f"(median {obs_table['n_obs'].median():.0f})"
)
obs_table

## Create the full catalogue

Start from `figures/figures_medellin/asteroid_analysis_results.csv`. Add the mean absolute magnitudes and colors in `colors.parquet`, the SBDB physical properties (numbers without units), and the orbital elements in `obs_final_elements_geometry_H.parquet` at the epoch of each asteroid's first observation. A missing value stays NaN.

In [10]:
from pathlib import Path

analysis_path = Path("figures/figures_medellin/asteroid_analysis_results.csv")
catalogue = pd.read_csv(analysis_path)
catalogue["provid"] = catalogue["provid"].astype(str).str.strip()
provids = catalogue["provid"].tolist()

color_columns = [
    "provid", "H_u", "H_g", "H_r", "H_i", "H_z", "H_y",
    "N_u", "N_g", "N_r", "N_i", "N_z", "N_y",
    "u-g", "g-r", "r-i", "i-z", "z-y",
]
colors = (
    pq.read_table(
        data_dir + "colors.parquet",
        columns=color_columns,
        filters=[("provid", "in", provids)],
    )
    .to_pandas()
    .drop_duplicates("provid")
)
catalogue = catalogue.merge(colors, on="provid", how="left")

def parse_quantity(value):
    """Number without its unit, or the original text when it is not a single number."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    match = re.fullmatch(
        r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)"
        r"(?:\s*(?:km3\s*/\s*s2|g\s*/\s*cm3|km|h))?",
        text,
        flags=re.IGNORECASE,
    )
    if match:
        return float(match.group(1))
    return text

sbdb = pd.read_csv(data_dir + "sbdb_physical_properties.csv", low_memory=False)
sbdb["provid"] = sbdb["provid"].astype(str).str.strip()
sbdb = sbdb[sbdb["provid"].isin(provids)].copy()
sbdb["value_clean"] = sbdb["value"].map(parse_quantity)
sbdb["sigma_clean"] = sbdb["sigma"].map(parse_quantity)

parameters = ["H", "diameter", "rot_per", "albedo", "GM", "density", "BV", "UB", "spec_B", "spec_T", "extent", "pole"]
values = (
    sbdb.pivot(index="provid", columns="parameter", values="value_clean")
    .reindex(columns=parameters)
    .rename(columns=lambda name: f"{name}_sbdb")
)
sigmas = (
    sbdb.pivot(index="provid", columns="parameter", values="sigma_clean")
    .reindex(columns=parameters)
    .rename(columns=lambda name: f"{name}_sbdb_err")
)
sbdb_wide = values.join(sigmas).reset_index()
sbdb_wide = sbdb_wide[["provid"] + [column for name in parameters for column in (f"{name}_sbdb", f"{name}_sbdb_err")]]
catalogue = catalogue.merge(sbdb_wide, on="provid", how="left")

designations = pd.read_csv(data_dir + "mpc_designations.csv")
designations["provid"] = designations["provid"].astype(str).str.strip()
designations = designations.drop_duplicates("provid")
names = catalogue[["provid"]].merge(designations, on="provid", how="left")
geometry = (
    pq.read_table(
        data_dir + "obs_final_elements_geometry_H.parquet",
        columns=["obstime", "mpc_designation", "a", "e", "q", "incl", "node", "peri", "M"],
        filters=[("mpc_designation", "in", names["mpc_designation"].dropna().unique().tolist())],
    )
    .to_pandas()
    .sort_values("obstime")
)
# After sorting by time, the first row of each asteroid is its first observation.
elements = (
    geometry.groupby("mpc_designation", as_index=False, sort=False)
    .first()
    .rename(columns={"obstime": "epoch_tdb"})
)
elements = names.merge(elements, on="mpc_designation", how="left")
catalogue = catalogue.merge(elements, on="provid", how="left")
catalogue["permid"] = catalogue["mpc_designation"].astype(str).str.extract(r"^\((\d+)\)", expand=False).astype("Int64")
catalogue["period_best"] = pd.to_numeric(catalogue["periods_h"].astype(str).str.split(",").str[0], errors="coerce")
catalogue = catalogue.sort_values("provid").reset_index(drop=True)

out = "medellin-vroc.csv"
catalogue.to_csv(out, index=False)
print(f"{len(catalogue)} asteroids, {catalogue.shape[1]} columns -> {out}")
print(f"  colors: {catalogue['H_r'].notna().sum()}")
print(f"  SBDB H: {catalogue['H_sbdb'].notna().sum()}")
print(f"  orbital elements: {catalogue['a'].notna().sum()}")
catalogue.head()


508 asteroids, 72 columns -> medellin-vroc.csv
  colors: 498
  SBDB H: 508
  orbital elements: 508


,provid,periods_h,amp_lsm,amp_photo,chi_r_sph,q_sph,q_sph_err,phi_sph_deg,phi_sph_err_deg,amp_sph,...,epoch,a,e,q,incl,node,peri,M,permid,period_best
0,1978 UU1,"2.935525,2.759569,2.608601,3.135448,2.468720,2...",0.317339,0.326436,2.721491,0.568896,0.005035,75.703056,0.780153,0.304427,...,2025-04-28 06:10:02.125,2.394876,0.225102,1.855784,3.02971,127.11233,241.28386,219.223085,4273,2.935525
1,1981 EY23,"15.032604,11.429361,21.953803,9.219493,7.72572...",0.592225,0.599374,27.933484,0.775576,0.020398,90.000000,NaN,0.511330,...,2025-04-22 02:37:21.011,2.679461,0.178664,2.200739,11.45531,170.71264,93.34151,308.677658,90687,15.032604
2,1986 PX5,"10.128747,12.737879,8.354165,17.610244,25.7670...",0.670653,0.658647,565.290394,0.999998,1529.212192,89.999124,NaN,0.752118,...,2025-04-25 02:50:23.667,2.545664,0.242333,1.928765,5.09751,112.20190,275.80501,157.346136,4996,10.128747
3,1992 EP8,"2.794221,2.634336,2.974769,3.172703,2.496416,2...",0.102418,0.111396,2.464047,0.293421,0.006174,68.279437,2.725687,0.089671,...,2025-04-28 06:21:30.929,2.204587,0.113423,1.954535,4.01987,75.53643,139.24435,0.570683,35121,2.794221
4,1997 JU3,"4.498158,4.110404,3.794952,4.966690",0.532402,0.553867,75.730926,0.749751,0.036133,90.000000,NaN,0.484291,...,2025-04-22 02:38:05.222,2.808641,0.155121,2.372961,8.95369,56.03704,156.69037,346.792442,22495,4.498158


In [11]:
print_list(catalogue.columns)

provid, periods_h, amp_lsm, amp_photo, chi_r_sph, q_sph, q_sph_err, phi_sph_deg,
phi_sph_err_deg, amp_sph, m_mid_sph, m_sep, p_rot_sph_h, chi_r_ell, b_over_a,
b_over_a_err, amp_ell, m_mid_ell, m_ab, p_rot_ell_h, H_u, H_g, H_r, H_i, H_z,
H_y, N_u, N_g, N_r, N_i, N_z, N_y, u-g, g-r, r-i, i-z, z-y, H_sbdb, H_sbdb_err,
diameter_sbdb, diameter_sbdb_err, rot_per_sbdb, rot_per_sbdb_err, albedo_sbdb,
albedo_sbdb_err, GM_sbdb, GM_sbdb_err, density_sbdb, density_sbdb_err, BV_sbdb,
BV_sbdb_err, UB_sbdb, UB_sbdb_err, spec_B_sbdb, spec_B_sbdb_err, spec_T_sbdb,
spec_T_sbdb_err, extent_sbdb, extent_sbdb_err, pole_sbdb, pole_sbdb_err,
mpc_designation, epoch, a, e, q, incl, node, peri, M, permid, period_best


### Catalogue columns

In [12]:
"""
Columns:

Identification
provid, permid, mpc_designation, 

Orbit
epoch_tdb, a, e, q, incl, node, peri, M,

Photometría
H_u, H_g, H_r, H_i, H_z, H_y, N_u, N_g, N_r, N_i, N_z, N_y, u-g, g-r, r-i, i-z, z-y,

Rotation
periods_h, period_best, amp_lsm, amp_photo, 

Shape
q_sph, q_sph_err, phi_sph_deg, phi_sph_err_deg, chi_r_sph, amp_sph, m_mid_sph, m_sep, p_rot_sph_h, 

SBDB
b_over_a, b_over_a_err, amp_ell, m_mid_ell, m_ab, p_rot_ell_h, chi_r_ell, 

SBDB physical parameters
H_sbdb, H_sbdb_err, albedo_sbdb, albedo_sbdb_err, 
diameter_sbdb, diameter_sbdb_err, extent_sbdb, extent_sbdb_err, pole_sbdb, pole_sbdb_err, density_sbdb, density_sbdb_err, 
rot_per_sbdb, rot_per_sbdb_err, 
GM_sbdb, GM_sbdb_err, BV_sbdb,
BV_sbdb_err, UB_sbdb, UB_sbdb_err, 
spec_B_sbdb, spec_B_sbdb_err, spec_T_sbdb,
spec_T_sbdb_err, 

Dynamical
res_body, res_flag, res_type, res_kp, res_k, res_orbit
"""

'\nColumns:\n\nIdentification\nprovid, permid, mpc_designation, \n\nOrbit\nepoch, a, e, q, incl, node, peri, M,\n\nPhotometría\nH_u, H_g, H_r, H_i, H_z, H_y, N_u, N_g, N_r, N_i, N_z, N_y, u-g, g-r, r-i, i-z, z-y,\n\nRotation\nperiods_h, period_best, amp_lsm, amp_photo, \n\nShape\nq_sph, q_sph_err, phi_sph_deg, phi_sph_err_deg, chi_r_sph, amp_sph, m_mid_sph, m_sep, p_rot_sph_h, \n\nSBDB\nb_over_a, b_over_a_err, amp_ell, m_mid_ell, m_ab, p_rot_ell_h, chi_r_ell, \n\nSBDB physical parameters\nH_sbdb, H_sbdb_err, albedo_sbdb, albedo_sbdb_err, \ndiameter_sbdb, diameter_sbdb_err, extent_sbdb, extent_sbdb_err, pole_sbdb, pole_sbdb_err, density_sbdb, density_sbdb_err, \nrot_per_sbdb, rot_per_sbdb_err, \nGM_sbdb, GM_sbdb_err, BV_sbdb,\nBV_sbdb_err, UB_sbdb, UB_sbdb_err, \nspec_B_sbdb, spec_B_sbdb_err, spec_T_sbdb,\nspec_T_sbdb_err, \n\nDynamical\nres_body, res_flag, res_type, res_kp, res_k, res_orbit\n'

In [13]:
catalogue["permid"] = catalogue["mpc_designation"].astype(str).str.extract(r"^\((\d+)\)", expand=False).astype("Int64")
catalogue["period_best"] = pd.to_numeric(catalogue["periods_h"].astype(str).str.split(",").str[0], errors="coerce")

catalogue_columns = [
    # Identification
    "provid", "permid", "mpc_designation",
    # Orbit
    "epoch_tdb", "a", "e", "q", "incl", "node", "peri", "M",
    # Photometry
    "H_u", "H_g", "H_r", "H_i", "H_z", "H_y",
    "N_u", "N_g", "N_r", "N_i", "N_z", "N_y",
    "u-g", "g-r", "r-i", "i-z", "z-y",
    # Rotation
    "periods_h", "period_best", "amp_lsm", "amp_photo",
    # Shape
    "q_sph", "q_sph_err", "phi_sph_deg", "phi_sph_err_deg", "chi_r_sph",
    "amp_sph", "m_mid_sph", "m_sep", "p_rot_sph_h",
    # Ellipsoid
    "b_over_a", "b_over_a_err", "amp_ell", "m_mid_ell", "m_ab", "p_rot_ell_h", "chi_r_ell",
    # SBDB physical parameters
    "H_sbdb", "H_sbdb_err", "albedo_sbdb", "albedo_sbdb_err",
    "diameter_sbdb", "diameter_sbdb_err", "extent_sbdb", "extent_sbdb_err",
    "pole_sbdb", "pole_sbdb_err", "density_sbdb", "density_sbdb_err",
    "rot_per_sbdb", "rot_per_sbdb_err",
    "GM_sbdb", "GM_sbdb_err", "BV_sbdb", "BV_sbdb_err", "UB_sbdb", "UB_sbdb_err",
    "spec_B_sbdb", "spec_B_sbdb_err", "spec_T_sbdb", "spec_T_sbdb_err",
    # Dynamical
    "res_body", "res_flag", "res_type", "res_kp", "res_k", "res_orbit",
]

catalogue = catalogue.reindex(columns=catalogue_columns)
catalogue.to_csv(out, index=False)
print_list(catalogue.columns)

provid, permid, mpc_designation, epoch, a, e, q, incl, node, peri, M, H_u, H_g,
H_r, H_i, H_z, H_y, N_u, N_g, N_r, N_i, N_z, N_y, u-g, g-r, r-i, i-z, z-y,
periods_h, period_best, amp_lsm, amp_photo, q_sph, q_sph_err, phi_sph_deg,
phi_sph_err_deg, chi_r_sph, amp_sph, m_mid_sph, m_sep, p_rot_sph_h, b_over_a,
b_over_a_err, amp_ell, m_mid_ell, m_ab, p_rot_ell_h, chi_r_ell, H_sbdb,
H_sbdb_err, albedo_sbdb, albedo_sbdb_err, diameter_sbdb, diameter_sbdb_err,
extent_sbdb, extent_sbdb_err, pole_sbdb, pole_sbdb_err, density_sbdb,
density_sbdb_err, rot_per_sbdb, rot_per_sbdb_err, GM_sbdb, GM_sbdb_err, BV_sbdb,
BV_sbdb_err, UB_sbdb, UB_sbdb_err, spec_B_sbdb, spec_B_sbdb_err, spec_T_sbdb,
spec_T_sbdb_err, res_body, res_flag, res_type, res_kp, res_k, res_orbit


In [14]:
catalogue.head()

,provid,permid,mpc_designation,epoch,a,e,q,incl,node,peri,...,spec_B_sbdb,spec_B_sbdb_err,spec_T_sbdb,spec_T_sbdb_err,res_body,res_flag,res_type,res_kp,res_k,res_orbit
0,1978 UU1,4273,(4273) Dunhuang,2025-04-28 06:10:02.125,2.394876,0.225102,1.855784,3.02971,127.11233,241.28386,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1981 EY23,90687,(90687) 1981 EY23,2025-04-22 02:37:21.011,2.679461,0.178664,2.200739,11.45531,170.71264,93.34151,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1986 PX5,4996,(4996) Veisberg,2025-04-25 02:50:23.667,2.545664,0.242333,1.928765,5.09751,112.20190,275.80501,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1992 EP8,35121,(35121) 1992 EP8,2025-04-28 06:21:30.929,2.204587,0.113423,1.954535,4.01987,75.53643,139.24435,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1997 JU3,22495,(22495) Fubini,2025-04-22 02:38:05.222,2.808641,0.155121,2.372961,8.95369,56.03704,156.69037,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
catalogue.iloc[0].period_best, catalogue.iloc[0].periods_h

(np.float64(2.935525),
 '2.935525,2.759569,2.608601,3.135448,2.468720,2.343078,3.364593,3.741774,4.072788,3.226372,3.469514,3.620029,3.015076')

## Column documentation

One row of `medellin-vroc.csv` is one asteroid. A missing value is NaN. Magnitudes in the rotation and shape blocks are absolute magnitudes in the LSST r band (`Lr`).

### Identification

| Column | Unit | Meaning |
|---|---|---|
| `provid` | | Provisional MPC designation, `YYYY LLnn`. |
| `permid` | | Permanent number: the integer in parentheses in `mpc_designation`. NaN when the asteroid is not numbered. |
| `mpc_designation` | | MPC designation used for the geometry. Numbered objects look like `(4273) Dunhuang`; unnumbered ones keep the provisional designation. |

### Orbit

Elements at `epoch_tdb`, taken from the row of `obs_final_elements_geometry_H.parquet` at the first observation.

| Column | Unit | Meaning |
|---|---|---|
| `epoch_tdb` | TDB | Epoch of the elements, equal to the time of the first observation. |
| `a` | au | Semimajor axis. |
| `e` | | Eccentricity. |
| `q` | au | Perihelion distance. |
| `incl` | deg | Inclination to the ecliptic. |
| `node` | deg | Longitude of the ascending node. |
| `peri` | deg | Argument of perihelion. |
| `M` | deg | Mean anomaly. |

### Photometry

Mean absolute magnitudes and colors from `colors.parquet`.

| Column | Unit | Meaning |
|---|---|---|
| `H_u`, `H_g`, `H_r`, `H_i`, `H_z`, `H_y` | mag | Mean absolute magnitude in that LSST band. |
| `N_u`, `N_g`, `N_r`, `N_i`, `N_z`, `N_y` | | Number of observations in that mean. |
| `u-g`, `g-r`, `r-i`, `i-z`, `z-y` | mag | Color, the difference of the two absolute magnitudes. |

### Rotation

Lomb–Scargle fit of the absolute magnitude in `Lr`. Periods at or above 80% of the main peak are kept, strongest first.

| Column | Unit | Meaning |
|---|---|---|
| `periods_h` | h | Those rotation periods, separated by commas. |
| `period_best` | h | First value of `periods_h`. |
| `amp_lsm` | mag | Peak-to-peak amplitude of the periodogram model over one cycle. |
| `amp_photo` | mag | Width of the photometry between the 5th and 95th percentiles. |

### Shape: touching spheres

Two spheres in contact, fitted to the folded `Lr` light curve. `q_sph` is reported with the smaller sphere second, so it is at most 1.

| Column | Unit | Meaning |
|---|---|---|
| `q_sph` | | Radius ratio `r2/r1`. |
| `q_sph_err` | | Uncertainty of `q_sph`. |
| `phi_sph_deg` | deg | Angle between the spin axis and the line of sight. |
| `phi_sph_err_deg` | deg | Symmetric uncertainty of that angle. NaN when the fit only gives a lower limit. |
| `chi_r_sph` | | Reduced chi-squared of the fit. |
| `amp_sph` | mag | Peak-to-peak amplitude of the model. |
| `m_mid_sph` | mag | Midpoint of the model curve, halfway between its brightest and faintest magnitudes. |
| `m_sep` | mag | Magnitude when the two disks are fully separated. |
| `p_rot_sph_h` | h | Rotation period refined by the fit. |

### Shape: ellipsoid

Equator-on triaxial ellipsoid (`phi = 90°`) rotating about its short axis. The long axis is `a = 1`.

| Column | Unit | Meaning |
|---|---|---|
| `b_over_a` | | Axis ratio `b/a`. |
| `b_over_a_err` | | Uncertainty of `b/a`. |
| `amp_ell` | mag | Peak-to-peak amplitude of the model. |
| `m_mid_ell` | mag | Midpoint of the model curve. |
| `m_ab` | mag | Magnitude of the `a`–`b` face, the brightest aspect when seen equator-on. |
| `p_rot_ell_h` | h | Rotation period refined by the fit. |
| `chi_r_ell` | | Reduced chi-squared of the fit. |

### SBDB physical parameters

From the JPL Small-Body Database. The number is stored without its unit. Each quantity has an `_err` column; that uncertainty is NaN when SBDB does not give a numeric error. Taxonomic classes have no numeric error.

| Column | Unit | Meaning |
|---|---|---|
| `H_sbdb` | mag | Absolute magnitude. |
| `albedo_sbdb` | | Geometric albedo. |
| `diameter_sbdb` | km | Diameter. |
| `extent_sbdb` | km | Axis extents, kept as text `a x b` or `a x b x c`. |
| `pole_sbdb` | deg | Ecliptic pole, kept as text `longitude/latitude`. |
| `density_sbdb` | g cm⁻³ | Bulk density. |
| `rot_per_sbdb` | h | Rotation period in the literature. |
| `GM_sbdb` | km³ s⁻² | Gravitational parameter. |
| `BV_sbdb` | mag | Color B−V. |
| `UB_sbdb` | mag | Color U−B. |
| `spec_B_sbdb` | | Bus taxonomic class. |
| `spec_T_sbdb` | | Tholen taxonomic class. |

### Dynamics

Reserved for a mean-motion resonance. They are not computed yet, so every value is NaN.

| Column | Unit | Meaning |
|---|---|---|
| `res_body` | | Planet the asteroid is resonant with. |
| `res_flag` | | Whether a resonance is identified. |
| `res_type` | | Kind of resonance. |
| `res_kp` | | One integer of the resonance. |
| `res_k` | | The other integer of the resonance. |
| `res_orbit` | | Description of the resonant orbit. |
